Сначала надо поставить в secrets HF_TOKEN <токен hugging face>

In [ ]:
# Установка базовых пакетов (без указания версий)
!pip install -q torch torchvision torchaudio
!pip install -q transformers accelerate bitsandbytes
!pip install -q sentence-transformers

# Проверка установленных версий
import torch, transformers, bitsandbytes, sentence_transformers
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Bitsandbytes: {bitsandbytes.__version__}")
print(f"CUDA доступен: {torch.cuda.is_available()}")

In [ ]:
# Авторизация в Hugging Face
from huggingface_hub import notebook_login
notebook_login()

# Подключение Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Загрузка модели с 4-битной квантизацией
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Конфигурация квантизации
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Загрузка модели Mistral-7B
model_name = "mistralai/Mistral-7B-v0.1"
try:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Сохранение модели на Google Drive
    save_path = "/content/drive/MyDrive/mistral-7b"
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Модель сохранена в: {save_path}")
    
except Exception as e:
    print(f"Ошибка: {e}")
    print("\nЕсли не хватает памяти, попробуйте 8-битную версию:")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        load_in_8bit=True,
        device_map="auto"
    )

In [ ]:
# Проверка работы модели
input_text = "Объясни, как работает квантизация в нейросетях"
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

app = FastAPI()

# Загружаем модель (если она уже сохранена в Colab)
model_path = "/content/drive/MyDrive/mistral-7b"
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

class Query(BaseModel):
    text: str

@app.post("/generate")
def generate_text(query: Query):
    inputs = tokenizer(query.text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_length=100)

    # # Проверка работы модели
    # input_text = "Объясни, как работает квантизация в нейросетях"
    # inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    # outputs = model.generate(**inputs, max_new_tokens=100)
    # print(tokenizer.decode(outputs[0], skip_special_tokens=True))

    return {"response": tokenizer.decode(outputs[0], skip_special_tokens=True)}

In [ ]:
!pip install pyngrok uvicorn nest_asyncio

In [ ]:
!ngrok authtoken 1k14aQW4Fs5ExlpfpHDMOCHqENC_6tAfL5QFav5wnbhNhDLy7

In [ ]:
from pyngrok import ngrok
import uvicorn
import nest_asyncio

ngrok_tunnel = ngrok.connect(8000)  # Открываем туннель к порту 8000
print("Public URL:", ngrok_tunnel.public_url)  # Копируем этот URL для FastAPI

nest_asyncio.apply()  # Для работы в Colab
uvicorn.run(app, host="0.0.0.0", port=8000)  # Запускаем сервер